In [1]:
import pandas as pd
import os

In [2]:
from pathlib import Path
import traceback

def add_filename_features(df, filename):
    try:
        parts = Path(filename).stem.split('_')
        df = df.copy()

        keywords = ['control', 'exo', 'endo']

        split_idx = next(
            (i for i, p in enumerate(parts) if any(k in p.lower() for k in keywords)),
            len(parts)
        )

        df['position'] = '_'.join(parts[:split_idx])

        if split_idx < len(parts):
            df['class'] = parts[split_idx]
            start_idx = split_idx + 1
        else:
            start_idx = split_idx

        index_param = 0
        names = ['group', 'nm', 'center', 'obj', 'power', 'during', 'acc', 'map', 'step', 'place']
        for i, part in enumerate(parts[start_idx:], start=start_idx + 1):

            if index_param >= 10:
                df[names[-1]] = df[names[-1]] + '_' + part.replace(names[-1], '')
            else:
                df[names[index_param]] = part.replace(names[index_param], '')
            index_param += 1

        return df

    except Exception as e:
        print("\nERROR in add_filename_features")
        print("filename:", filename)
        print("exception type:", type(e).__name__)
        print("message:", e)
        print("\nTraceback:")
        traceback.print_exc()

        raise

In [3]:

def convert_spectra_file_to_csv(input_filepath, output_filepath=None):
    # Читаем файл с пропуском строк, начинающихся с #
    df_raw = pd.read_csv(input_filepath, sep='\t', comment='#',
                         names=['X', 'Y', 'Wave', 'Intensity'])

    unique_waves = sorted(df_raw['Wave'].unique(), reverse=True)
    n_waves = len(unique_waves)

    print(f"Найдено уникальных волн: {n_waves}")
    print(f"Диапазон волн: от {unique_waves[-1]:.3f} до {unique_waves[0]:.3f}")

    # Создаем сводную таблицу
    df_pivot = df_raw.pivot_table(
        index=['X', 'Y'],
        columns='Wave',
        values='Intensity',
        aggfunc='first'  # Берем первое значение
    )

    df_result = df_pivot.reset_index()

    df_result = add_filename_features(df_result, input_filepath)




    meta_cols = list(df_result.columns[-12:])
    xy_cols = [df_result.columns[0], df_result.columns[1]]
    wave_cols = list(df_result.columns[2:-12])
    
    new_order = meta_cols + xy_cols + wave_cols
    
    df_result = df_result[new_order]
    
    df_result.columns = (
        meta_cols +
        xy_cols +
        [f'Wave_{float(col):.1f}' for col in wave_cols]
    )
    

    # Сохраняем в CSV
    if output_filepath is None:
        input_path = Path(input_filepath)
        output_filepath = input_path.with_suffix('.csv')

    df_result.to_csv(output_filepath, index=False)
    print(f"Файл сохранен: {output_filepath}")
    print(f"Размерность результирующего dataframe: {df_result.shape}")

    return df_result


In [4]:
def convert_dataset_structure(input_dir='dataset', output_dir='datasetCsv'):
    # Рекурсивно преобразует все .txt файлы из input_dir в .csv файлы в output_dir,
    # сохраняя структуру директорий


    # Преобразуем в Path объекты
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    # Проверяем, существует ли исходная директория
    if not input_dir.exists():
        print(f"Ошибка: Директория {input_dir} не существует!")
        return

    print(f"Начинаем обработку файлов из {input_dir}")
    print(f"Результаты будут сохранены в {output_dir}")
    print("-" * 50)

    # для статистики
    total_txt_files = 0
    converted_files = 0
    skipped_files = 0
    errors = 0

    # Рекурсивно обходим все файлы в исходной директории
    for root, dirs, files in os.walk(input_dir):
        root_path = Path(root)

        relative_path = root_path.relative_to(input_dir)

        target_dir = output_dir / relative_path
        target_dir.mkdir(parents=True, exist_ok=True)

        # Обрабатываем каждый .txt файл в текущей директории
        for file in files:
            if file.endswith('.txt'):
                total_txt_files += 1

                # Формируем полные пути
                input_file = root_path / file
                output_file = target_dir / file.replace('.txt', '.csv')

                try:
                    convert_spectra_file_to_csv(input_file, output_file)
                    converted_files += 1

                except Exception as e:
                    print(f"✗ Ошибка при преобразовании {input_file}: {str(e)}")
                    errors += 1

            if not file.endswith('.txt'):
                skipped_files += 1

    print("-" * 25)
    print(f"Статистика обработки:")
    print(f"  Всего найдено .txt файлов: {total_txt_files}")
    print(f"  Успешно преобразовано: {converted_files}")
    print(f"  Пропущено (не .txt файлы): {skipped_files}")
    print(f"  Ошибок: {errors}")

    if converted_files == total_txt_files and errors == 0:
        print("\n✓ Все файлы успешно преобразованы!")

In [5]:

if __name__ == "__main__":
    convert_dataset_structure('dataset', 'datasetCsv')

Начинаем обработку файлов из dataset
Результаты будут сохранены в datasetCsv
--------------------------------------------------
Найдено уникальных волн: 1015
Диапазон волн: от 926.755 до 2002.418
Файл сохранен: datasetCsv\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_1.csv
Размерность результирующего dataframe: (525, 1029)
Найдено уникальных волн: 1015
Диапазон волн: от 926.755 до 2002.418
Файл сохранен: datasetCsv\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_2.csv
Размерность результирующего dataframe: (525, 1029)
Найдено уникальных волн: 1015
Диапазон волн: от 926.755 до 2002.418
Файл сохранен: datasetCsv\control\mk1\cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_1.csv
Размерность результирующего dataframe: (525, 1029)
Найдено уникальных волн: 1015
Диапазон волн: от 926.755 до 2002.418
Файл сохранен: datasetCsv\control\mk1\cortex_control_1group_

In [6]:

df = pd.read_csv("datasetCsv/control/mk1/cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place5_1.csv")

df.head()

,position,class,group,nm,center,obj,power,during,acc,map,...,Wave_3281.5,Wave_3282.2,Wave_3283.0,Wave_3283.7,Wave_3284.5,Wave_3285.2,Wave_3286.0,Wave_3286.7,Wave_3287.4,Wave_3288.2
0,cortex,control,1,633,2900,100,100,1s,5,35x15,...,3976.281006,4118.689941,3954.133057,4029.101563,4117.598633,3976.580078,4098.863281,4038.836182,4046.338623,4019.317871
1,cortex,control,1,633,2900,100,100,1s,5,35x15,...,4802.564941,4496.489746,4628.899902,4558.891602,4674.486816,4742.864746,4706.602539,4660.195801,4694.833496,4799.538574
2,cortex,control,1,633,2900,100,100,1s,5,35x15,...,4698.014648,4789.958984,4790.843750,4764.733398,4745.363281,4604.460938,4747.118164,4680.457520,4843.446777,4921.131348
3,cortex,control,1,633,2900,100,100,1s,5,35x15,...,4785.702148,4793.332031,4952.787598,4923.333008,4876.991699,4911.650391,4963.203125,4977.629395,4941.396973,4745.497070
4,cortex,control,1,633,2900,100,100,1s,5,35x15,...,5686.183105,5619.769043,5641.049805,5770.321289,5771.387695,5661.056641,5820.791016,5828.621582,5623.667480,5654.065918


In [7]:

df = pd.read_csv("datasetCsv/control/mk1/cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_1.csv")

df.head()


,position,class,group,nm,center,obj,power,during,acc,map,...,Wave_1993.8,Wave_1994.7,Wave_1995.7,Wave_1996.7,Wave_1997.6,Wave_1998.6,Wave_1999.5,Wave_2000.5,Wave_2001.5,Wave_2002.4
0,cortex,control,1,633,1500,100,100,1s,5,35x15,...,12658.952148,12875.486328,13037.258789,12694.952148,12942.969727,13076.103516,12984.517578,12979.162109,13013.024414,12803.853516
1,cortex,control,1,633,1500,100,100,1s,5,35x15,...,10330.268555,10418.412109,10545.763672,10594.799805,10573.313477,10632.827148,10415.318359,10281.377930,10419.309570,10437.611328
2,cortex,control,1,633,1500,100,100,1s,5,35x15,...,9753.319336,9773.462891,10046.941406,9724.959961,9988.084961,10023.967773,9903.046875,10035.650391,10021.884766,9919.914063
3,cortex,control,1,633,1500,100,100,1s,5,35x15,...,9165.926758,9102.402344,9370.529297,9348.813477,9175.556641,9305.356445,9176.457031,9251.411133,9098.919922,9245.338867
4,cortex,control,1,633,1500,100,100,1s,5,35x15,...,8899.642578,8846.510742,8950.057617,8938.708984,8885.554688,9030.978516,8860.208008,8825.307617,8918.510742,8936.812500


In [8]:
def count_files_in_dirs(root_dir):
    root = Path(root_dir)

    for d in root.rglob('*'):
        if d.is_dir():
            files = [f for f in d.iterdir() if f.is_file()]
            if files:
                print(f"{d}: {len(files)} files")

In [9]:
count_files_in_dirs("datasetCsv")

datasetCsv\control\mk1: 12 files
datasetCsv\control\mk2a: 24 files
datasetCsv\control\mk2b: 24 files
datasetCsv\control\mk3: 20 files
datasetCsv\endo\mend1: 13 files
datasetCsv\endo\mend2a: 24 files
datasetCsv\endo\mend2b: 24 files
datasetCsv\endo\mend3: 12 files
datasetCsv\exo\mexo1: 12 files
datasetCsv\exo\mexo2a: 24 files
datasetCsv\exo\mexo2b: 24 files
datasetCsv\exo\mexo3: 24 files
